# Implement FlashAttention-2 in Triton — Solution

**Difficulty**: 🟣 Expert

**Companies**: NVIDIA, Meta, Together AI, xAI

---

### Problem Statement

Standard attention computes `softmax(Q @ K^T / sqrt(d)) @ V`, which requires **materializing the full N x N attention matrix**. For sequence length N=8192 and batch*heads=32, this matrix alone takes 8192² × 32 × 4 bytes ≈ 8.6 GB in float32. Standard attention is **O(N^2) in memory**, and because those N×N scores are written to and re-read from HBM, it is also memory-bandwidth bound.

**FlashAttention** (Dao et al., 2022) and its successor **FlashAttention-2** solve this by:
1. **Tiling**: Processing Q, K, V in blocks rather than all at once
2. **Online softmax**: Maintaining running softmax statistics (max and sum) across blocks
3. **Never materializing the full N x N matrix**: score tiles live in on-chip SRAM, so extra memory is O(N) instead of O(N^2), and HBM traffic drops sharply (the main source of the speedup)

The algorithm processes tiles of Q against ALL tiles of K and V, accumulating the attention output using the online softmax trick to correctly combine partial results from different K,V blocks. The result is **exact** attention, not an approximation.

What FlashAttention-2 changes relative to FlashAttention-1:
- **Loop order**: Q tiles in the outer loop, K/V tiles in the inner loop. Each Q tile's running statistics stay on-chip for the whole inner loop, and different Q tiles run in parallel (one program per (batch·head, Q tile)).
- **Deferred normalization**: keep the output *unnormalized* inside the loop (only rescale by `exp(m_old − m_new)`) and divide by the running sum `l` once at the end.
- **Causal masking**: skip K/V tiles that lie entirely in the future of the Q tile (about half the work), and mask only the tiles on the diagonal.

This notebook implements the **forward pass**. Autograd through the PyTorch loop below would store every P tile, which is O(N²) memory again. The real kernel saves only the output and the row-wise log-sum-exp `L = m + log(l)`, and its backward pass recomputes the tiles.

Your task:
1. Implement FlashAttention-2 in **pure PyTorch** (the tiled algorithm with online softmax), with optional causal masking
2. Write the Triton kernel version (runs on a GPU, or on CPU through Triton's interpreter)
3. Show that the output matches standard attention to float tolerance
4. Demonstrate the memory advantage

---

### Requirements

1. **Tiled Processing** — Split Q, and K/V, into tiles of `block_size` rows (one size for both here; real kernels tune `BLOCK_Q` and `BLOCK_KV` separately).
2. **Online Softmax** — For each Q block, iterate over all K,V blocks, maintaining `running_max` and `running_sum` to correctly combine softmax across K blocks.
3. **No Full Matrix** — Never create an N x N tensor. The largest intermediate should be `block_size_q x block_size_kv`.
4. **Correctness** — Output must match `torch.softmax(Q @ K^T / sqrt(d), dim=-1) @ V` within float32 tolerance.
5. **Causal masking** — `causal=True` must match `F.scaled_dot_product_attention(Q, K, V, is_causal=True)`. Skip K/V tiles that are entirely in the future, and mask positions `k > q` inside tiles that straddle the diagonal.

---

### Constraints

- ✅ Pure PyTorch implementation must work on CPU
- ✅ Triton kernel is optional: it needs CUDA, or `TRITON_INTERPRET=1` set before importing triton to run it on CPU (slow, but the same code)
- ✅ Must work for any sequence length (not just multiples of block_size)
- ❌ Do **not** materialize the full N x N attention matrix in the flash implementation

---

<details>
  <summary>💡 Hint</summary>

  **Online softmax across blocks:**
  
  When processing Q_block against K_block_j:
  ```
  S_j = Q_block @ K_block_j^T / sqrt(d)           # (block_q, block_kv)
  block_max_j = S_j.max(dim=-1, keepdim=True)      # max within this block
  new_max = max(running_max, block_max_j)           # global running max
  
  # Correction factor for previously accumulated results
  correction = exp(running_max - new_max)
  
  # Update running sum and output
  P_j = exp(S_j - new_max)                         # local attention weights
  running_sum = running_sum * correction + P_j.sum(dim=-1, keepdim=True)
  running_output = running_output * correction + P_j @ V_block_j
  running_max = new_max
  ```
  
  After all K,V blocks: `output = running_output / running_sum`

  **In the Triton kernel**, keys past the end of the sequence are loaded as zeros, so their scores come out as 0, not −∞. Mask them to `-inf` explicitly, or they leak into `running_sum` whenever N is not a multiple of the block size.

</details>

---

In [ ]:
import torch
import torch.nn.functional as F
import math
import os

# No GPU? Uncomment to run the Triton kernel on CPU with Triton's interpreter
# (must be set before triton is imported):
# os.environ["TRITON_INTERPRET"] = "1"

TRITON_AVAILABLE = False
try:
    import triton
    import triton.language as tl
    TRITON_AVAILABLE = True
    print("Triton is available!")
except ImportError:
    print("Triton not available. Will use pure PyTorch implementation only.")

CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA available: {CUDA_AVAILABLE}")

# Triton kernels run on a GPU, or on CPU through Triton's interpreter
# (TRITON_INTERPRET=1 must be set before importing triton; slow, but handy for debugging).
TRITON_INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
RUN_TRITON = TRITON_AVAILABLE and (CUDA_AVAILABLE or TRITON_INTERPRET)
TRITON_DEVICE = "cuda" if CUDA_AVAILABLE else "cpu"

In [3]:
# Test data
torch.manual_seed(42)

# Dimensions
batch_size = 2
n_heads = 4
seq_len = 128
head_dim = 64

# Create Q, K, V tensors: (batch, heads, seq_len, head_dim)
Q = torch.randn(batch_size, n_heads, seq_len, head_dim)
K = torch.randn(batch_size, n_heads, seq_len, head_dim)
V = torch.randn(batch_size, n_heads, seq_len, head_dim)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"\nFull attention matrix would be: {batch_size} x {n_heads} x {seq_len} x {seq_len}")
print(f"= {batch_size * n_heads * seq_len * seq_len * 4 / 1024:.1f} KB")

Q shape: torch.Size([2, 4, 128, 64])
K shape: torch.Size([2, 4, 128, 64])
V shape: torch.Size([2, 4, 128, 64])

Full attention matrix would be: 2 x 4 x 128 x 128
= 512.0 KB


In [4]:
def standard_attention(Q, K, V):
    """
    Standard attention: O(N^2) memory.
    This is the reference implementation.
    
    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)  # (B, H, N, N) - full matrix!
    attn_weights = torch.softmax(scores, dim=-1)       # (B, H, N, N)
    output = attn_weights @ V                          # (B, H, N, d)
    return output

In [ ]:
def flash_attention_pytorch(Q, K, V, block_size=32, causal=False):
    """
    FlashAttention-2 forward pass in pure PyTorch.

    Processes attention in tiles, never materializing the full N x N matrix.
    Uses online softmax to correctly combine results across K,V blocks.

    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
        block_size: tile size for both Q and K,V blocks
        causal: if True, query i only attends to keys j <= i
    Returns:
        output: (batch, heads, seq_len, head_dim)
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    acc = dict(dtype=Q.dtype, device=Q.device)
    out_tiles = []

    # Outer loop over Q tiles (FA2 order: each tile's statistics stay on-chip)
    for q_start in range(0, N, block_size):
        q_end = min(q_start + block_size, N)
        Q_block = Q[:, :, q_start:q_end, :]  # (B, H, block_q, d)
        block_q = q_end - q_start

        # Online softmax accumulators for this Q tile
        running_max = torch.full((B, H, block_q, 1), float('-inf'), **acc)
        running_sum = torch.zeros((B, H, block_q, 1), **acc)
        running_output = torch.zeros((B, H, block_q, d), **acc)  # unnormalized

        # Causal: K,V tiles starting after this Q tile's last row are all future
        kv_stop = q_end if causal else N
        for kv_start in range(0, kv_stop, block_size):
            kv_end = min(kv_start + block_size, N)
            K_block = K[:, :, kv_start:kv_end, :]  # (B, H, block_kv, d)
            V_block = V[:, :, kv_start:kv_end, :]  # (B, H, block_kv, d)

            # Local scores: the ONLY score matrix we build, (B, H, block_q, block_kv)
            S = Q_block @ K_block.transpose(-2, -1) * scale

            # Tile straddles the diagonal: hide keys that come after the query
            if causal and kv_end - 1 > q_start:
                q_pos = torch.arange(q_start, q_end, device=Q.device)[:, None]
                k_pos = torch.arange(kv_start, kv_end, device=Q.device)[None, :]
                S = S.masked_fill(k_pos > q_pos, float('-inf'))

            # Online softmax update. Every row has seen key 0 in the first tile,
            # so running_max is finite from then on and no exp(-inf - -inf) occurs.
            new_max = torch.maximum(running_max, S.amax(dim=-1, keepdim=True))
            correction = torch.exp(running_max - new_max)  # rescales old sum AND output
            P = torch.exp(S - new_max)                       # unnormalized weights
            running_sum = running_sum * correction + P.sum(dim=-1, keepdim=True)
            running_output = running_output * correction + P @ V_block
            running_max = new_max

        # FA2: normalize once, after the inner loop
        out_tiles.append(running_output / running_sum)

    return torch.cat(out_tiles, dim=2)

In [ ]:
if TRITON_AVAILABLE:
    @triton.jit
    def flash_attention_kernel(
        Q, K, V, Out,
        stride_qbh, stride_qn, stride_qd,
        stride_kbh, stride_kn, stride_kd,
        stride_vbh, stride_vn, stride_vd,
        stride_obh, stride_on, stride_od,
        N, scale,
        D: tl.constexpr,
        BLOCK_Q: tl.constexpr,
        BLOCK_KV: tl.constexpr,
        CAUSAL: tl.constexpr,
    ):
        """One program = one (batch*head, Q tile). It streams over the K/V tiles,
        keeping the running max m, running sum l and unnormalized output acc on-chip."""
        q_block = tl.program_id(0)
        bh = tl.program_id(1)

        q_rows = q_block * BLOCK_Q + tl.arange(0, BLOCK_Q)  # query positions of this tile
        offs_kv = tl.arange(0, BLOCK_KV)
        offs_d = tl.arange(0, D)
        q_valid = q_rows < N

        q = tl.load(Q + bh * stride_qbh + q_rows[:, None] * stride_qn + offs_d[None, :] * stride_qd,
                    mask=q_valid[:, None], other=0.0)

        m_i = tl.full((BLOCK_Q,), float("-inf"), dtype=tl.float32)
        l_i = tl.zeros((BLOCK_Q,), dtype=tl.float32)
        acc = tl.zeros((BLOCK_Q, D), dtype=tl.float32)

        # Causal: K/V tiles that start after this Q tile's last row are all future
        hi = tl.minimum(N, (q_block + 1) * BLOCK_Q) if CAUSAL else N
        for kv_start in range(0, hi, BLOCK_KV):
            kv_rows = kv_start + offs_kv
            kv_valid = kv_rows < N
            k = tl.load(K + bh * stride_kbh + kv_rows[:, None] * stride_kn + offs_d[None, :] * stride_kd,
                        mask=kv_valid[:, None], other=0.0)
            v = tl.load(V + bh * stride_vbh + kv_rows[:, None] * stride_vn + offs_d[None, :] * stride_vd,
                        mask=kv_valid[:, None], other=0.0)

            s = tl.dot(q, tl.trans(k)) * scale  # (BLOCK_Q, BLOCK_KV), fp32
            # Keys past N were loaded as zeros and would score 0, not -inf, leaking
            # into l_i: mask them (and, if causal, keys after the query) explicitly.
            keep = kv_valid[None, :]
            if CAUSAL:
                keep = keep & (kv_rows[None, :] <= q_rows[:, None])
            s = tl.where(keep, s, float("-inf"))

            m_new = tl.maximum(m_i, tl.max(s, axis=1))
            alpha = tl.exp(m_i - m_new)  # rescales the old l and acc
            p = tl.exp(s - m_new[:, None])
            l_i = l_i * alpha + tl.sum(p, axis=1)
            acc = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v)
            m_i = m_new

        out = acc / l_i[:, None]  # FA2: normalize once, at the end
        tl.store(Out + bh * stride_obh + q_rows[:, None] * stride_on + offs_d[None, :] * stride_od,
                 out.to(Out.dtype.element_ty), mask=q_valid[:, None])

    def flash_attention_triton(Q, K, V, block_size=32, causal=False):
        """Launch one program per (Q tile, batch*head). Q, K, V: (B, H, N, D) on CUDA,
        or on CPU under TRITON_INTERPRET=1."""
        B, H, N, D = Q.shape
        assert D >= 16 and D & (D - 1) == 0, "head_dim must be a power of two >= 16 (tl.arange / tl.dot)"
        assert block_size >= 16, "tl.dot needs tiles of at least 16"
        # View (B, H, N, D) as (B*H, N, D) so one grid axis covers every (batch, head)
        q, k, v = (t.reshape(B * H, N, D) for t in (Q, K, V))
        out = torch.empty_like(q)
        grid = (triton.cdiv(N, block_size), B * H)
        flash_attention_kernel[grid](
            q, k, v, out,
            q.stride(0), q.stride(1), q.stride(2),
            k.stride(0), k.stride(1), k.stride(2),
            v.stride(0), v.stride(1), v.stride(2),
            out.stride(0), out.stride(1), out.stride(2),
            N, 1.0 / math.sqrt(D),
            D=D, BLOCK_Q=block_size, BLOCK_KV=block_size, CAUSAL=causal,
        )
        return out.view(B, H, N, D)
    print("Triton flash attention kernel defined.")
else:
    print("Triton not available. The kernel code above shows the GPU implementation.")
    print("The pure PyTorch flash_attention_pytorch is the main implementation to study.")

In [ ]:
# Validation
print("=" * 60)
print("VALIDATING FLASH ATTENTION")
print("=" * 60)

# Reference output
ref_output = standard_attention(Q, K, V)

# Flash attention output
flash_output = flash_attention_pytorch(Q, K, V, block_size=32)

# Test 1: Correctness
print("\n--- Correctness Test ---")
max_err = (flash_output - ref_output).abs().max().item()
print(f"Max absolute error: {max_err:.2e}")
assert torch.allclose(flash_output, ref_output, atol=1e-5), "Flash attention output MISMATCH!"
print("PASSED")

# Test 2: Different block sizes
print("\n--- Block Size Robustness Test ---")
for bs in [16, 32, 64]:
    flash_out = flash_attention_pytorch(Q, K, V, block_size=bs)
    err = (flash_out - ref_output).abs().max().item()
    assert torch.allclose(flash_out, ref_output, atol=1e-5), f"Failed for block_size={bs}"
    print(f"  block_size={bs}: max_err={err:.2e} PASSED")

# Test 3: Non-divisible sequence length
print("\n--- Non-Divisible Sequence Length Test ---")
Q2 = torch.randn(1, 1, 50, 32)
K2 = torch.randn(1, 1, 50, 32)
V2 = torch.randn(1, 1, 50, 32)
ref2 = standard_attention(Q2, K2, V2)
flash2 = flash_attention_pytorch(Q2, K2, V2, block_size=16)
err2 = (flash2 - ref2).abs().max().item()
assert torch.allclose(flash2, ref2, atol=1e-5), "Non-divisible seq len FAILED!"
print(f"  seq_len=50, block_size=16: max_err={err2:.2e} PASSED")

# Test 4: Causal masking
print("\n--- Causal Masking Test ---")
for (n, bs) in [(seq_len, 32), (50, 16), (50, 64)]:
    Qc, Kc, Vc = Q[:, :, :n], K[:, :, :n], V[:, :, :n]
    ref_c = F.scaled_dot_product_attention(Qc, Kc, Vc, is_causal=True)
    flash_c = flash_attention_pytorch(Qc, Kc, Vc, block_size=bs, causal=True)
    err_c = (flash_c - ref_c).abs().max().item()
    assert torch.allclose(flash_c, ref_c, atol=1e-5), f"Causal FAILED for seq_len={n}, block_size={bs}"
    print(f"  seq_len={n}, block_size={bs}: max_err={err_c:.2e} PASSED")

# Test 5: Triton kernel (GPU, or CPU via TRITON_INTERPRET=1)
print("\n--- Triton Kernel Test ---")
if RUN_TRITON:
    for (n, bs, causal) in [(seq_len, 32, False), (100, 32, False), (100, 32, True)]:
        q_t, k_t, v_t = (t[:, :, :n].to(TRITON_DEVICE) for t in (Q, K, V))
        ref_t = F.scaled_dot_product_attention(q_t, k_t, v_t, is_causal=causal)
        out_t = flash_attention_triton(q_t, k_t, v_t, block_size=bs, causal=causal)
        err_t = (out_t - ref_t).abs().max().item()
        # tl.dot may use TF32 on GPUs, hence the loose tolerance
        assert torch.allclose(out_t, ref_t, atol=1e-2), f"Triton kernel FAILED (N={n}, causal={causal})"
        print(f"  N={n}, block={bs}, causal={causal}: max_err={err_t:.2e} PASSED")
else:
    print("  Skipped (needs Triton plus CUDA or TRITON_INTERPRET=1)")

# Test 6: Memory comparison
print("\n--- Memory Analysis ---")
N = seq_len
d = head_dim
bs = 32
standard_mem = N * N  # full attention matrix
flash_mem_actual = bs * bs  # actual peak: one tile of scores
print(f"  Standard attention peak: {N}x{N} = {standard_mem} elements")
print(f"  Flash attention peak:    {bs}x{bs} = {flash_mem_actual} elements per tile (plus O(block_q * d) accumulators)")
print(f"  Memory reduction: {standard_mem / flash_mem_actual:.1f}x")

# For longer sequences, the advantage is even more dramatic
print("\n--- Scaling Analysis ---")
for N_test in [256, 1024, 4096, 8192]:
    standard = N_test * N_test * 4  # bytes in float32
    flash = bs * bs * 4
    print(f"  N={N_test:>5d}: standard={standard/1024/1024:>8.1f} MB, flash_tile={flash/1024:.1f} KB, ratio={standard // flash}x")

print("\nAll tests passed!")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
# To also verify your Triton kernel (GPU, or CPU with TRITON_INTERPRET=1), add
# flash_attention_triton=flash_attention_triton to the call.
check("flash-attention-triton", flash_attention_pytorch=flash_attention_pytorch)